# 10.1 · 图像处理基础 / Image Processing Basics

> **课程定位 / Where this fits**
> 第 1 课，**Part 10 · 计算机视觉**。这是整个 CV 的地基。
> Lesson 1, **Part 10 · Computer Vision**. The foundation of all CV.
>
> 在让神经网络"看懂"图像之前，得先搞清楚：**一张图在计算机里到底是什么？** 答案是——**一堆数字(像素)排成的数组**。理解了"图像=数组"，后面的卷积、CNN 就都是在这个数组上做数学运算。本课用**大量可视化**带你看清像素、通道、色彩空间，以及**卷积/滤波**这个 CNN 的核心操作。
> Before a neural net can "see" images, we must understand: **what is an image to a computer?** Answer — **an array of numbers (pixels)**. Once you grasp "image = array," convolutions and CNNs are just math on that array. This lesson uses **lots of visualization** to make pixels, channels, color spaces, and **convolution/filtering** (the core CNN operation) crystal clear.
>
> 💼 **实战/面试视角**："图像怎么表示 / RGB vs 灰度 / 卷积是什么 / 边缘检测原理" 是 CV 入门必问。
> 💼 **Practical/interview angle:** "how images are represented / RGB vs grayscale / what is convolution / edge detection" — CV entry must-knows.

> 📐 **符号约定 / Notation**
> - 图像形状 $(H, W)$ 或 $(H, W, C)$ —— 高、宽、通道数 / height, width, channels
> - 像素值 —— 通常 0~255(uint8) 或 0~1(float) / pixel value, usually 0–255 or 0–1
> - 卷积核(kernel/filter) —— 一个小矩阵, 在图上滑动做加权求和 / a small matrix slid over the image

> 💡 **面试相关 / Interview-relevant**
> - "彩色图像怎么存储(H×W×C)"（出镜率 ★★★★）
> - "卷积操作具体怎么算"（★★★★★，CNN 基础）
> - "为什么要转灰度/换色彩空间"（★★★）
> - "边缘检测/Sobel 原理"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解数字图像 = 像素数组，掌握形状/通道/取值范围。
   Understand a digital image = pixel array; grasp shape/channels/value range.
2. 会在 RGB / 灰度 / HSV 之间转换并理解用途。
   Convert between RGB / grayscale / HSV and know why.
3. 把图像当数组做基本操作（裁剪、翻转、调亮度、阈值）。
   Manipulate images as arrays (crop, flip, brightness, threshold).
4. **从零实现卷积**，彻底理解滤波（模糊/锐化）。
   Implement convolution from scratch to truly understand filtering (blur/sharpen).
5. 用 Sobel 做边缘检测，为 CNN 学卷积核埋下直觉。
   Do edge detection with Sobel, seeding intuition for CNN-learned kernels.

## 目录 / TOC
1. [图像是什么：像素与通道 ⭐](#1)
2. [色彩空间：RGB / 灰度 / HSV ⭐](#2)
3. [把图像当数组：基本操作 ⭐](#3)
4. [卷积与滤波（从零实现）⭐](#4)
5. [边缘检测 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 图像是什么：像素与通道 ⭐ / What Is an Image: Pixels & Channels

一张数字图像就是一个**数字网格**。每个格子是一个**像素(pixel)**，存着这个位置的颜色/亮度。
A digital image is a **grid of numbers**. Each cell is a **pixel** holding the color/brightness at that spot.
- **灰度图(grayscale)**：形状 $(H, W)$，每个像素一个数（0=黑，255=白）。
  **Grayscale:** shape $(H, W)$, one number per pixel (0=black, 255=white).
- **彩色图(RGB)**：形状 $(H, W, 3)$，每个像素**三个数**——红、绿、蓝三个**通道(channel)**的强度，叠加出各种颜色。
  **Color (RGB):** shape $(H, W, 3)$, **three numbers** per pixel — Red/Green/Blue **channel** intensities that combine into colors.

> ⚠️ **易错点(面试/实战)**：PyTorch/卷积里图像常用 **$(C, H, W)$**（通道在前），而 matplotlib/skimage 用 **$(H, W, C)$**（通道在后）。两者要会互转（`permute`/`transpose`），否则报错或图像错乱。
> ⚠️ **Gotcha:** PyTorch/conv use **$(C, H, W)$** (channels-first); matplotlib/skimage use **$(H, W, C)$** (channels-last). Convert between them (`permute`/`transpose`) or you'll get errors/garbled images.

下面加载一张彩色图和一张灰度图，打印它们的"真身"——数组。
Let's load a color and a grayscale image and print their true form — arrays.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import data, color, filters
sns.set_theme(style="white")

astro = data.astronaut()        # 彩色图: (H, W, 3) uint8 / color image
cam = data.camera()             # 灰度图: (H, W) uint8 / grayscale image
print(f"彩色图 astronaut: shape={astro.shape}, dtype={astro.dtype}, 取值范围[{astro.min()}, {astro.max()}]")
print(f"灰度图 camera:    shape={cam.shape}, dtype={cam.dtype}, 取值范围[{cam.min()}, {cam.max()}]")
print(f"\n左上角 3×3 像素的灰度值(就是一堆数字):\n{cam[:3, :3]}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(astro); axes[0].set_title(f"彩色图 RGB {astro.shape}"); axes[0].axis("off")
axes[1].imshow(cam, cmap="gray"); axes[1].set_title(f"灰度图 {cam.shape}"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print("\n图像的本质 = 像素数组; 灰度=1个数/像素, 彩色=RGB 3个数/像素")


In [ ]:
# 把彩色图拆成 R/G/B 三个通道分别可视化 / split into R/G/B channels
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(astro); axes[0].set_title("原彩色图"); axes[0].axis("off")
for i, (name, cmap) in enumerate([("R 红通道","Reds"), ("G 绿通道","Greens"), ("B 蓝通道","Blues")]):
    axes[i+1].imshow(astro[:, :, i], cmap=cmap)        # astro[:,:,i] 取第 i 个通道(0=R,1=G,2=B) / select channel i
    axes[i+1].set_title(f"{name} 越亮值越大"); axes[i+1].axis("off")
plt.tight_layout(); plt.show()
print("每个通道是一张灰度图(该颜色的强度); 三通道叠加 → 彩色")
print(f"中心像素 RGB 值 = {astro[256, 256]}  (分别是红/绿/蓝强度, 0~255)")


<a id="2"></a>
## 2. 色彩空间：RGB / 灰度 / HSV ⭐ / Color Spaces

**色彩空间**是表示颜色的不同"坐标系"。同一张图可以换不同表示，各有用途：
A **color space** is a different "coordinate system" for color. The same image can be re-expressed for different purposes:
- **灰度(grayscale)**：丢掉颜色只留亮度。很多任务（边缘、纹理）不需要颜色，转灰度能**省 2/3 计算**、简化问题。注意：不是简单三通道求平均，而是按人眼敏感度加权（绿>红>蓝）。
  **Grayscale:** drop color, keep brightness. Many tasks (edges, texture) don't need color; grayscale **cuts 2/3 of compute**. Note: it's a perceptual weighted sum (G>R>B), not a plain average.
- **HSV(色相/饱和度/明度)**：把"是什么颜色(H)"和"多鲜艳(S)""多亮(V)"**分开**。做**按颜色筛选**（如"找出所有红色区域"）时，HSV 比 RGB 方便得多——这是实战常用技巧。
  **HSV (Hue/Saturation/Value):** separates "which color (H)" from "how vivid (S)" and "how bright (V)." For **color-based selection** (e.g. "find all red regions"), HSV is far easier than RGB — a common practical trick.


In [ ]:
gray = color.rgb2gray(astro)        # RGB→灰度(感知加权): 返回 0~1 float / perceptual grayscale, 0..1
hsv = color.rgb2hsv(astro)          # RGB→HSV / convert to HSV
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(gray, cmap="gray"); axes[0].set_title("灰度(感知加权)"); axes[0].axis("off")
for i, name in enumerate(["H 色相(什么颜色)", "S 饱和度(多鲜艳)", "V 明度(多亮)"]):
    im = axes[i+1].imshow(hsv[:, :, i], cmap="viridis")   # HSV 三通道 / the three HSV channels
    axes[i+1].set_title(name); axes[i+1].axis("off")
plt.tight_layout(); plt.show()
print("灰度: 0.299R+0.587G+0.114B (绿色权重最大, 因人眼对绿最敏感)")
print("HSV: H=颜色种类, S=鲜艳度, V=明暗; 按颜色找物体时用 H 通道阈值最方便")


<a id="3"></a>
## 3. 把图像当数组：基本操作 ⭐ / Images as Arrays: Basic Ops

既然图像就是 NumPy 数组，很多"图像处理"其实就是**数组操作**。这也是为什么数据增强(10.5)能用几行代码搞定。
Since an image is a NumPy array, much "image processing" is just **array ops**. That's why data augmentation (10.5) takes only a few lines.
- **裁剪(crop)** = 数组切片
  **Crop** = array slicing
- **翻转(flip)** = 数组反向索引
  **Flip** = reverse indexing
- **调亮度(brightness)** = 整体加/乘一个数
  **Brightness** = add/multiply a constant
- **二值化(threshold)** = 比较运算
  **Threshold** = comparison


In [ ]:
g = cam.astype(float) / 255.0       # 转到 0~1 方便运算 / scale to 0..1
crop = g[50:250, 150:400]            # 裁剪: 行50~250, 列150~400 / crop via slicing
flip = g[:, ::-1]                    # 水平翻转: 列方向反向 / horizontal flip (reverse columns)
bright = np.clip(g + 0.3, 0, 1)      # 调亮: +0.3 再截断到[0,1] / brighten then clip
thresh = (g > 0.5).astype(float)     # 二值化: 大于0.5为白(1)否则黑(0) / threshold to black/white

fig, axes = plt.subplots(1, 5, figsize=(14, 3.2))
for ax, im, t in zip(axes, [g, crop, flip, bright, thresh],
                     ["原图", "裁剪(切片)", "水平翻转", "调亮(+0.3)", "二值化(>0.5)"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=1); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print("图像=数组 → 裁剪=切片, 翻转=反向索引, 调亮=加常数, 二值化=比较; 都是纯数组运算")


<a id="4"></a>
## 4. 卷积与滤波（从零实现）⭐ / Convolution & Filtering (From Scratch)

**这是全 Part 10 最重要的概念**——CNN 的 "C" 就是卷积(Convolution)。
**This is the most important concept in all of Part 10** — the "C" in CNN is Convolution.

**卷积**：拿一个小矩阵叫**卷积核(kernel/filter)**（比如 3×3），让它在图像上**逐位置滑动**；每到一个位置，就把核盖住的那一小块像素与核**对应相乘再求和**，得到输出图该位置的一个值。
**Convolution:** take a small matrix called a **kernel/filter** (e.g. 3×3) and **slide it over every position** of the image; at each spot, **multiply the covered pixels by the kernel elementwise and sum**, producing one output value there.

不同的核做不同的事：
Different kernels do different things:
- 核全是 $1/9$（3×3）→ **求邻域平均** → **模糊(blur)**（去噪/平滑）。
  All $1/9$ (3×3) → **neighborhood average** → **blur** (denoise/smooth).
- 中间大、周围负 → **锐化(sharpen)**（强调差异）。
  Big center, negative around → **sharpen** (emphasize differences).
- 一边正一边负 → **检测边缘**（找亮度突变）。
  Positive vs negative sides → **edge detection** (find brightness jumps).

**关键认知(面试)**：传统图像处理里这些核是**人手工设计**的；而 **CNN 的革命在于让网络自己从数据里学出最有用的卷积核**！理解了这里手写的卷积，就理解了 CNN 每一层在干什么。
**Key insight (interview):** in classical CV these kernels are **hand-designed**; the **CNN revolution is letting the network learn the best kernels from data**! Understand this hand-written convolution and you understand what every CNN layer does.

下面**从零实现**卷积（不调库），亲手算一遍。
Below we implement convolution **from scratch** (no library) to compute it by hand.


In [ ]:
def convolve2d(img, kernel):
    """对灰度图做 2D 卷积(valid, 不补零) / 2D convolution on a grayscale image."""
    kh, kw = kernel.shape
    H, W = img.shape
    out = np.zeros((H - kh + 1, W - kw + 1))          # 输出比输入小(没补零) / output is smaller (no padding)
    for i in range(out.shape[0]):                      # 遍历每个输出位置(行) / each output row
        for j in range(out.shape[1]):                  # 遍历每个输出位置(列) / each output col
            patch = img[i:i+kh, j:j+kw]                # 核盖住的小块 / the patch under the kernel
            out[i, j] = np.sum(patch * kernel)         # 对应相乘再求和 = 卷积核心 / multiply-and-sum
    return out

box = np.ones((9, 9)) / 81.0                          # 9×9 均值核 → 模糊 / box blur kernel
sharpen = np.array([[0,-1,0], [-1,5,-1], [0,-1,0]], float)  # 锐化核(中心5,四邻-1) / sharpen kernel

blurred = convolve2d(g, box)
sharpened = np.clip(convolve2d(g, sharpen), 0, 1)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, im, t in zip(axes, [g, blurred, sharpened], ["原图", "模糊(9×9均值核)", "锐化(中心5,邻-1)"]):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print("卷积 = 核在图上滑动, 每处'对应相乘再求和'; 这就是 CNN 每个卷积层做的事")
print("均值核→模糊(邻域平均消除细节); 锐化核→强调中心与邻域的差异")
print("⚠️ CNN 不手工设计核, 而是从数据学出最有用的核(这是关键区别)")


In [ ]:
# 不同模糊强度: 核越大/高斯 sigma 越大 → 越模糊 / stronger blur with larger kernel / sigma
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(g, cmap="gray"); axes[0].set_title("原图(清晰)"); axes[0].axis("off")
for ax, sigma in zip(axes[1:], [1, 3, 6]):
    blur = filters.gaussian(g, sigma=sigma)            # 高斯模糊: sigma 越大越糊 / Gaussian blur
    ax.imshow(blur, cmap="gray"); ax.set_title(f"高斯模糊 σ={sigma}"); ax.axis("off")
plt.tight_layout(); plt.show()
print("高斯模糊比均值模糊更自然(按距离加权: 越近权重越大); σ 控制模糊程度")
print("实战用途: 去噪 / 降采样前抗锯齿 / 突出大结构忽略细节")


<a id="5"></a>
## 5. 边缘检测 + 小结 ⭐ / Edge Detection & Summary

**边缘**就是图像里**亮度突然变化**的地方（物体轮廓）。怎么找突变？——**求梯度(导数)**。**Sobel 算子**就是两个专门求水平/垂直方向亮度变化的卷积核。
An **edge** is where **brightness changes sharply** (object outlines). How to find changes? **Take the gradient (derivative).** The **Sobel operator** is two convolution kernels for horizontal/vertical brightness change.

把水平梯度 $G_x$ 和垂直梯度 $G_y$ 合起来 $\sqrt{G_x^2 + G_y^2}$ 就得到**边缘强度图**。这正是 CNN 浅层经常自动学到的东西（9.14 里我们见过第一层学到的"模板"很多就是边缘检测器）。
Combine horizontal $G_x$ and vertical $G_y$ as $\sqrt{G_x^2 + G_y^2}$ to get an **edge-strength map**. This is exactly what CNN early layers often learn automatically (in 9.14 many first-layer "templates" were edge detectors).


In [ ]:
# Sobel 边缘检测: 分别求水平/垂直梯度再合成 / Sobel edge detection
sobel_x = filters.sobel_h(g)          # 水平方向亮度变化(检测水平边缘) / horizontal gradient
sobel_y = filters.sobel_v(g)          # 垂直方向亮度变化(检测垂直边缘) / vertical gradient
edges = np.sqrt(sobel_x**2 + sobel_y**2)   # 梯度幅值 = 总边缘强度 / gradient magnitude

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, im, t in zip(axes, [g, sobel_x, sobel_y, edges],
                     ["原图", "Gx 水平梯度", "Gy 垂直梯度", "边缘强度 √(Gx²+Gy²)"]):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print("边缘=亮度突变处; Sobel 用卷积核求梯度; Gx+Gy 合成得到完整边缘图")
print("呼应 9.14: CNN 浅层会自动学到类似 Sobel 的边缘检测核 → 卷积核可学是 CNN 的威力")


```
图像=像素数组: 灰度(H,W)1数/像素; 彩色(H,W,3) RGB 3数/像素; 值0~255或0~1
通道顺序: PyTorch/卷积 (C,H,W) 通道在前; matplotlib/skimage (H,W,C) 通道在后 → 要会互转
色彩空间: 灰度=感知加权(省2/3计算); HSV分离颜色/鲜艳/明暗(按颜色筛选方便)
图像处理=数组操作: 裁剪=切片, 翻转=反向索引, 调亮=加常数, 二值化=比较
卷积: 核在图上滑动, 每处'对应相乘求和'; 均值核→模糊, 锐化核, Sobel→边缘
CNN 核心: 不手工设计核, 而是从数据学出最有用的卷积核(对比传统CV)
边缘检测: 求梯度(Sobel); CNN浅层自动学到类似边缘检测器
```

### 💡 面试速查 / Interview cheat-sheet
1. **图像表示**: 彩色 (H,W,3)/(C,H,W), 灰度 (H,W); 注意通道顺序。
   Image rep: color (H,W,3)/(C,H,W), grayscale (H,W); mind channel order.
2. **卷积**: 核滑动+对应相乘求和; CNN 让核可学习。
   Convolution: slide kernel + multiply-sum; CNN makes kernels learnable.
3. **滤波核**: 均值→模糊, 锐化, Sobel→边缘检测。
   Filter kernels: box→blur, sharpen, Sobel→edges.
4. **色彩空间**: 灰度省算力; HSV 便于按颜色筛选。
   Color spaces: grayscale saves compute; HSV eases color selection.
5. **传统CV vs CNN**: 手工设计核 vs 数据驱动学核。
   Classical CV vs CNN: hand-designed vs data-learned kernels.

### 下一节 / Next
**10.2 卷积神经网络(CNN)**——把本课"卷积"的概念变成可学习的网络层。我们会从零实现卷积层、理解池化与感受野、在 FashionMNIST 上训练一个真 CNN，并**可视化它学到的卷积核和特征图**。
**10.2 CNN** — turn this lesson's "convolution" into learnable network layers. We'll implement a conv layer from scratch, understand pooling and receptive fields, train a real CNN on FashionMNIST, and **visualize its learned kernels and feature maps**.
